<div style="border-left:4px solid #818cf8;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#818cf8;">Architectures</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">Four designs for the same question, run and measured.</div></div>

<div style="font:400 15px/1.65 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#3f3f46;">Four ways of answering the same question, differing only in what leaves the machine. They share every node they have in common, so the comparison is between designs and not between four separate programs.</div>

In [ ]:
# The code comes from GitHub. The repository is private, so a fresh clone needs a
# GITHUB_TOKEN secret (Add-ons -> Secrets).
#
# A version created through the API cannot read that secret - Kaggle answers 400
# no matter how the box is ticked in the editor, and the attachment cannot be
# declared in kernel-metadata.json either (Kaggle/kaggle-cli#582). Notebook 1's
# output carries the whole repository, so fall back to that copy. Only notebook 1
# has no input to fall back to, and only notebook 1 has to be saved from the
# browser rather than pushed.
import os, shutil, subprocess, sys
from pathlib import Path

ROOT = Path("/kaggle/working/nl2sql")


def clone_from_github() -> str:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://github.com/Kirazul/NL2SQL-demo.git".replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", url, str(ROOT)], check=True)
    # git writes the clone URL into .git/config, token and all, and Kaggle saves
    # .git with the notebook output. Put the plain address back immediately.
    subprocess.run(["git", "-C", str(ROOT), "remote", "set-url", "origin", "https://github.com/Kirazul/NL2SQL-demo.git"], check=True)
    return "a fresh clone"


def copy_from_setup() -> str:
    marker = next(iter(sorted(Path("/kaggle/input").glob("*/nl2sql/pyproject.toml"))), None)
    if marker is None:
        return ""
    # Everything except data/ and models/: those are the two gigabytes that get
    # read where they are mounted and are never worth copying.
    shutil.copytree(marker.parent, ROOT,
                    ignore=shutil.ignore_patterns("data", "models", ".git"))
    # The mount is read-only and copytree keeps the modes, but `pip install -e .`
    # writes an egg-info back into the tree.
    subprocess.run(["chmod", "-R", "u+w", str(ROOT)], check=True)
    return f"the copy in {marker.parent}"


if ROOT.exists():
    source = "the working directory"
else:
    try:
        source = clone_from_github()
    except Exception as e:
        source = copy_from_setup()
        if not source:
            raise SystemExit(
                f"Nothing to run from. GITHUB_TOKEN could not be read "
                f"({type(e).__name__}: {e}) and notebook 1's output is not attached "
                "either. Check Add-ons -> Secrets, or add Input -> Your Work -> "
                "NL2SQL 1 Setup."
            ) from e
        print("GITHUB_TOKEN unreadable, falling back to notebook 1's output:", e)

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("code:", ROOT, "from", source)

In [ ]:
# pip writes to site-packages, which is not part of notebook 1's saved output, so
# every session installs again. Nearly all of it is already in the Kaggle image.
!pip install -q -e . 2>&1 | tail -2
print("dependencies ready")

In [ ]:
# Notebook 1 built the database, the index and the model weights and saved them
# with its output. Kaggle mounts that output read-only under /kaggle/input, and
# every one of the three is opened read-only here too - so point the settings at
# the mount rather than copying two gigabytes into the working directory.
#
# How deep inside the mount they sit depends on what notebook 1's working
# directory held when it was saved. Matching one guessed shape reported a good
# output as a missing database, so search the plausible depths instead, and when
# nothing turns up print what is actually mounted rather than assert a cause.
MOUNTS = sorted(p for p in Path("/kaggle/input").glob("*") if p.is_dir())
found = [
    hit
    for m in MOUNTS
    for pattern in ("data/eicu.db", "*/data/eicu.db", "*/*/data/eicu.db")
    for hit in sorted(m.glob(pattern))
]
if not found:
    print("mounted under /kaggle/input:" if MOUNTS else "nothing is mounted under /kaggle/input")
    for m in MOUNTS:
        print(" ", m.name + "/", " ".join(sorted(c.name for c in m.iterdir())[:8]))
    raise SystemExit(
        "No eicu.db under any of them. Notebook 1's saved output is what carries it: "
        "open the sidebar, Input -> Add Input -> Your Work -> NL2SQL 1 Setup, and check "
        "the version it pins is one that ran to the end."
    )
SETUP = found[0].parents[1]

# Name a missing piece here rather than several cells later, from inside whichever
# library opens it first.
for path in (SETUP / "data/index.db", SETUP / "models/gliner2-base-v1"):
    if not path.exists():
        print("missing from notebook 1's output:", path)

# Set before nl2sql is imported anywhere: settings() is read once and cached. A
# subprocess started later - the API server in notebook 5 - inherits these too.
os.environ["DB_PATH"] = str(SETUP / "data" / "eicu.db")
os.environ["INDEX_PATH"] = str(SETUP / "data" / "index.db")
os.environ["GLINER_MODEL"] = str(SETUP / "models" / "gliner2-base-v1")
weights = sorted(SETUP.glob("models/*/*.gguf"))
if weights:
    os.environ["LOCAL_GGUF_PATH"] = str(weights[0])

for name in ("DB_PATH", "INDEX_PATH", "GLINER_MODEL", "LOCAL_GGUF_PATH"):
    print(f"  {name:<16} {os.environ.get(name, 'missing - the steps that need it will say so')}")

In [ ]:
# Only the pages that run a model on this machine need llama-cpp-python, and
# building it costs several minutes. Notebook 1 keeps the wheel it built with its
# output, so installing from that is a copy. Runs after the cell above, so a mount
# that turned out to be unusable stops the notebook before the compile, not after.
wheelhouse = next(
    (w.parent for m in MOUNTS for pattern in ("wheels/*.whl", "*/wheels/*.whl")
     for w in m.glob(pattern)),
    None,
)
if wheelhouse:
    !pip install -q --find-links {wheelhouse} llama-cpp-python 2>&1 | tail -2
else:
    print("no wheel in notebook 1's output - building from source, a few minutes")
    !pip install -q llama-cpp-python 2>&1 | tail -2

try:
    import llama_cpp
    print("local model runtime: llama-cpp-python", llama_cpp.__version__)
except ImportError as e:
    print("llama-cpp-python is unavailable:", e)
    print("Full Local will fail, and the answer writer will show a plain table instead.")

In [ ]:
# Keys live in Kaggle secrets, never in the notebook.
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("GROQ_API_KEY", "OPENROUTER_API_KEY", "LANGSMITH_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"{name} not set - the steps that need it will say so")

os.environ["LANGSMITH_TRACING"] = "1"
os.environ["LANGSMITH_PROJECT"] = "nl2sql"

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#818cf8;">1.</span> The four designs</div></div>

<table style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;border-collapse:collapse;width:100%;">
<tr style="border-bottom:2px solid #e4e4e7;text-align:left;">
  <th style="padding:8px 10px;">Design</th><th style="padding:8px 10px;">Question</th>
  <th style="padding:8px 10px;">Schema</th><th style="padding:8px 10px;">Rows</th>
  <th style="padding:8px 10px;">Writes the SQL</th></tr>
<tr style="border-bottom:1px solid #f4f4f5;"><td style="padding:7px 10px;"><b>Full Cloud</b></td>
  <td style="padding:7px 10px;color:#dc2626;">sent as typed</td><td style="padding:7px 10px;color:#dc2626;">sent</td>
  <td style="padding:7px 10px;color:#dc2626;">sent</td><td style="padding:7px 10px;">cloud</td></tr>
<tr style="border-bottom:1px solid #f4f4f5;"><td style="padding:7px 10px;"><b>Hybrid</b></td>
  <td style="padding:7px 10px;color:#16a34a;">values hidden</td><td style="padding:7px 10px;color:#ca8a04;">sent</td>
  <td style="padding:7px 10px;color:#16a34a;">stay here</td><td style="padding:7px 10px;">cloud</td></tr>
<tr style="border-bottom:1px solid #f4f4f5;"><td style="padding:7px 10px;"><b>Hybrid Opaque</b></td>
  <td style="padding:7px 10px;color:#16a34a;">values hidden</td><td style="padding:7px 10px;color:#16a34a;">renamed t1, c7</td>
  <td style="padding:7px 10px;color:#16a34a;">stay here</td><td style="padding:7px 10px;">cloud</td></tr>
<tr><td style="padding:7px 10px;"><b>Full Local</b></td>
  <td style="padding:7px 10px;color:#16a34a;">never leaves</td><td style="padding:7px 10px;color:#16a34a;">never leaves</td>
  <td style="padding:7px 10px;color:#16a34a;">stay here</td><td style="padding:7px 10px;">this machine</td></tr>
</table>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#818cf8;">2.</span> Drawn from the code</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Each diagram is generated from the graph that runs, so it cannot describe something else.</div></div>

In [ ]:
from nl2sql.core import graph

print(graph.mermaid("hybrid"))

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#818cf8;">3.</span> One question, four times</div></div>

In [ ]:
import time
from nl2sql.core.state import ARMS

results = {}
for arm in ARMS:
    started = time.perf_counter()
    results[arm] = graph.run('How many patients over 65 received aspirin?', arm=arm, write=False)
    print(f"{arm:<15} {(time.perf_counter() - started):>6.1f} s")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#818cf8;">4.</span> What each one sent</div></div>

In [ ]:
print(f"{'design':<16}{'ok':<5}{'characters out':>15}{'real values out':>17}{'tokens':>9}")
for arm, state in results.items():
    print(f"{arm:<16}{str(state.get('success')):<5}"
          f"{state.get('egress_chars', 0):>15,}{state.get('egress_values', 0):>17}"
          f"{state.get('cloud_tokens', 0):>9}")

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;">The column that matters is <b>real values out</b>. It is zero for the three protected designs and not zero for the baseline, and no line of code asserts that — the gate measured it.</div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#818cf8;">5.</span> The queries they wrote</div></div>

In [ ]:
for arm, state in results.items():
    print(f"--- {arm}")
    print((state.get("sql") or "(none)")[:300])
    print(f"    rows: {state.get('row_count')}   {state.get('failure_reason', '')[:80]}\n")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#818cf8;">6.</span> What the opaque design showed the provider</div></div>

In [ ]:
opaque = results["hybrid_opaque"].get("opaque", {})
if opaque:
    print("question as sent:", opaque["question"])
    print("parameters      :", opaque["parameters"].strip())
    print()
    for label, real in opaque.get("labels", {}).items():
        print(f"  {label:<6} was {real}")